In [11]:
import anthropic
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_KEY"))

print("Client ready.")

Client ready.


In [4]:
import pickle

with open("../data/train_test_data.pkl", "rb") as f:
    data =pickle.load(f)

import pandas as pd
comparison_df = pd.read_csv("../outputs/model_comparison.csv", index_col= "Model")

print(comparison_df)

                              model       rmse       mae  directional_accuracy
Model                                                                         
LinearRegression  Linear Regression   2.067377  1.578433                 68.88
LSTM                           LSTM   8.021009  6.687082                 54.03
RandomForest           RandomForest  10.599141  8.051550                 63.90
XGBoost                     XGBoost  11.012762  8.554283                 67.22


In [5]:
import os
print(os.getcwd())

c:\Users\david\Downloads\Projects\Stock Machine Learning & AI\stock-forecast-ml-pipeline\notebooks


In [6]:
import os
for f in os.listdir("../data"):
    print(f)

train_test_data.pkl


In [7]:
def get_model_commentary(comparison_df):
    """
    Input: comparsion dataframe
    output: analyst-style commentary from Claude
    """
    #dataframe to string
    metrics_str = comparison_df.to_string()

    #THE PROMPT ENGINEERING

    prompt = f"""
    You are a quantitative analyst reviewing the results of a stock price forecasting experiment.
    Four models are trained on 5 years of AAPL historical data and evaluated on three metrics:
    - RMSE (Root Mean Square Error) - lower is better
    - MAE (Mean Average Value) - lower is better
    - Directional Accuracy - higher is better, 50% is a coin flip
    - For RMSE and MAE, these measure the difference between the actual and predicted stock prices

    Here are the results:
    {metrics_str}

    Write a concise analyst-style commentary (3-4 sentences) that:
    1. Identifies the best performing model and why
    2. Notes any interesting patterns between the models
    3. Comments on what the directional accuracy results mean practically
    Keep the tone professional but accessible to a non-technical audience. 
    Key ideas should be clearly conveyed.

"""
    message = client.messages.create(
        model= "claude-opus-4-5",
        max_tokens= 500,
        messages=[
            {"role" : "user", "content": prompt}
        ]
    )
    return(message.content[0].text)


In [ ]:
commentary = get_model_commentary(comparison_df)

## Model Performance Analysis

**Linear Regression emerges as the clear winner**, delivering the lowest prediction errors (RMSE of 2.07 and MAE of 1.58) while also achieving the highest directional accuracy at 68.88%—meaning it correctly predicted whether AAPL would move up or down nearly 7 out of 10 trading days. Interestingly, the more sophisticated models (LSTM, RandomForest, and XGBoost) significantly underperformed on error metrics, suggesting that AAPL's price movements over this period may follow relatively linear patterns that don't benefit from complex modeling approaches. From a practical trading perspective, while all models beat a coin flip on direction, Linear Regression's combination of tight price estimates and strong directional calls makes it the most actionable for investment decisions—though a 69% accuracy rate still means roughly 1 in 3 predictions will point the wrong way.


In [ ]:
display(Markdown(commentary))


## Model Performance Analysis

**Linear Regression emerges as the clear winner**, delivering the lowest prediction errors (RMSE of 2.07 and MAE of 1.58) while also achieving the highest directional accuracy at 68.88%—meaning it correctly predicted whether AAPL would move up or down nearly 7 out of 10 trading days. Interestingly, the more sophisticated models (LSTM, RandomForest, and XGBoost) significantly underperformed on error metrics, suggesting that AAPL's price movements over this period may follow relatively linear patterns that don't benefit from complex modeling approaches. From a practical trading perspective, while all models beat a coin flip on direction, Linear Regression's combination of tight price estimates and strong directional calls makes it the most actionable for investment decisions—though a 69% accuracy rate still means roughly 1 in 3 predictions will point the wrong way.

In [14]:
def get_per_model_commentary(model_name, metrics):
    """
    Input: model names and their metrics (MAE, RMSE and Direcrtional accuracy) as dict
    Output: analyst-style commentary for each model
    """
    prompt = f"""
    You are a seasoned quantitative analyst wrting a brief performace review for a single model
    in a stock price forecasting experiment

    Model:{model_name}
    RMSE : {metrics['rmse']} (lower is better, measures average dollar error)
    MSE : {metrics['mae']} (lower is better, average absolute dollar error)
    Directional Accuracy : {metrics['directional_accuracy']} ( higher is better and above 50% beats coin flip)

    Context: This model was one of four tested on 5 years of AAPL data.
    The best performing model achieved an RMSE of 2.07, MAE of 1.58, and 68.88% directional accuracy.

    Write 2-3 sentences that:
    1. Summarize how this model perfomed
    2. Compare is to the benchmarks above
    3. Give one practical insight about what these numbers mean

    Be direct and professional. No bullet points23, just prose
"""
    
    message = client.messages.create(
        model = 'claude-opus-4-5',
        max_tokens= 500,
        messages= [
            {"role" : "user", "content" : prompt}
        ]
    )
    return(message.content[0].text)

In [15]:
per_model_commentary ={}

for model_name, row in comparison_df.iterrows():
    print(f'Genterating commentary for {model_name}..')

    metrics = {
        "rmse" : row["rmse"],
        "mae" : row["mae"],
        "directional_accuracy" : row["directional_accuracy"]
    }
    model_commentary = get_per_model_commentary(model_name, metrics)
    per_model_commentary[model_name] = model_commentary
    print(f"Done.\n")

print(f'All commentaries generated')

Genterating commentary for LinearRegression..
Done.

Genterating commentary for LSTM..
Done.

Genterating commentary for RandomForest..
Done.

Genterating commentary for XGBoost..
Done.

All commentaries generated


In [17]:
print(per_model_commentary)

{'LinearRegression': '**Performance Review: Linear Regression Model**\n\nThe Linear Regression model delivered strong performance on the AAPL forecasting task, achieving an RMSE of $2.07, MAE of $1.58, and directional accuracy of 68.88%, which matches the best-performing model across all benchmarks in this experiment. These results indicate the model correctly predicts whether AAPL will move up or down roughly 7 out of 10 trading days, with typical prediction errors under $2—a meaningful edge for short-term trading strategies. The fact that a simple linear approach matched more complex alternatives suggests the underlying price relationships in this dataset may be largely linear, making this model an efficient and interpretable choice for production deployment.', 'LSTM': '**LSTM Performance Review**\n\nThe LSTM model delivered underwhelming results with an RMSE of $8.02 and directional accuracy of 54.03%, placing it well behind the top performer which achieved an RMSE of $2.07 and 68.8

In [18]:
with open("../outputs/per_model_commentary.json","w") as f:
    json.dump(per_model_commentary, f , indent= 4)

print("Per-model commentary saved to outputs/per_model_commentary.json")

overall ={"overall_commentary": commentary}

with open("../outputs/overall_commentary.json", "w") as f:
    json.dump(overall, f, indent= 4)

print("Overall commentary of the models saved to outputs/overall_commentary.json")

Per-model commentary saved to outputs/per_model_commentary.json
Overall commentary of the models saved to outputs/overall_commentary.json


In [19]:
with open("../outputs/per_model_commentary.json", "r") as f:
    saved = json.load(f)

for model, text in saved.items():
    print(f"\n {'='*50}")
    print(f"Model: {model}")
    print(f"f{'='*50}")
    print(text)


Model: LinearRegression
f==================================================
**Performance Review: Linear Regression Model**

The Linear Regression model delivered strong performance on the AAPL forecasting task, achieving an RMSE of $2.07, MAE of $1.58, and directional accuracy of 68.88%, which matches the best-performing model across all benchmarks in this experiment. These results indicate the model correctly predicts whether AAPL will move up or down roughly 7 out of 10 trading days, with typical prediction errors under $2—a meaningful edge for short-term trading strategies. The fact that a simple linear approach matched more complex alternatives suggests the underlying price relationships in this dataset may be largely linear, making this model an efficient and interpretable choice for production deployment.

Model: LSTM
f==================================================
**LSTM Performance Review**

The LSTM model delivered underwhelming results with an RMSE of $8.02 and directio